# Phase 1: Data Preprocessing, Copying, and Dataset Pipelines

In this phase, you will write the code to:
1. Copy the split image files from `data/raw` into structured directories under `data/processed/`.
2. Load these processed directories into TensorFlow `tf.data.Dataset` objects.
3. Implement data augmentation and dataset performance optimization (caching & prefetching).

### Checkpoint 1.1: Copy split files to `data/processed`

Write a function or loop that iterates through your split paths (`train_paths`, `val_paths`, `test_paths`) and copies the files to their respective destination in `data/processed/`.

**Expected Directory Structure:**
```text
data/processed/
├── train/
│   ├── Normal/
│   ├── Bacterial/
│   └── Viral/
├── val/
│   ├── Normal/
│   ├── Bacterial/
│   └── Viral/
└── test/
    ├── Normal/
    ├── Bacterial/
    └── Viral/
```

*Hint:* Use `os.makedirs(..., exist_ok=True)` to create the directories, and `shutil.copy(src, dst)` to copy files.

In [ ]:
# Write your code here to copy files from raw splits to data/processed splits
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. Define paths to where your raw data sits
raw_dir = '../data/raw/'
categories = ['NORMAL', 'PNEUMONIA']

all_filepaths = []
all_labels = []

# 2. Walk through raw folders and parse true multi-class labels from names
for folder in ['train', 'test', 'val']:
    for category in categories:
        folder_path = os.path.join(raw_dir, folder, category)
        if not os.path.exists(folder_path):
            continue
            
        for filename in os.listdir(folder_path):
            if filename.endswith('.jpeg') or filename.endswith('.png'):
                full_path = os.path.join(folder_path, filename)
                
                # Determine class based on folder and filename pattern
                if category == 'NORMAL':
                    label = 'Normal'
                elif 'bacteria' in filename.lower():
                    label = 'Bacterial'
                elif 'virus' in filename.lower():
                    label = 'Viral'
                else:
                    continue # Skip anomalies
                
                all_filepaths.append(full_path)
                all_labels.append(label)

# 3. Create your proper Stratified Splits (70% Train, 15% Val, 15% Test)
# This completely fixes the "16-image validation folder" issue from Kaggle
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_filepaths, all_labels, test_size=0.30, stratify=all_labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    test_paths, test_labels, test_size=0.50, stratify=test_labels, random_state=42
)

print(f"Train size: {len(train_paths)} | Val size: {len(val_paths)} | Test size: {len(test_paths)}")

processed_dir = '../data/processed'

# Define mapping for splits
splits = {
    'train': (train_paths, train_labels),
    'val': (val_paths, val_labels),
    'test': (test_paths, test_labels)
}

# Loop through each split and copy the files
for split_name, (paths, labels) in splits.items():
    print(f"Copying {split_name} split...")
    for path, label in zip(paths, labels):
        # Construct the target directory
        dest_dir = os.path.join(processed_dir, split_name, label)
        
        # Create the directory if it doesn't exist
        os.makedirs(dest_dir, exist_ok=True)
        
        # Copy the image from its raw path to the target directory
        shutil.copy(path, dest_dir)

print("All files copied successfully!")


Train size: 4099 | Val size: 878 | Test size: 879


### Checkpoint 1.2: Load directories into TensorFlow Datasets

Use `tf.keras.utils.image_dataset_from_directory` to load the `train`, `val`, and `test` subfolders from `data/processed/`.

Set the following parameters:
- `image_size=(224, 224)`
- `batch_size=32`
- `label_mode='categorical'` (to output one-hot encoded labels for our 3 classes)

In [ ]:
import tensorflow as tf

# Load train, val, and test datasets from directory
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
data_dir = '../'
# train_ds = tf.keras.utils.image_dataset_from_directory(...)
# val_ds = ...
# test_ds = ...


### Checkpoint 1.3: Define Data Augmentation layers

To fight overfitting, build a data augmentation pipeline using Keras preprocessing layers. Combine `RandomFlip`, `RandomRotation`, and `RandomZoom` layers into a `tf.keras.Sequential` block.

In [ ]:
# Create data augmentation sequential layers
# data_augmentation = tf.keras.Sequential([
#     tf.keras.layers.RandomFlip(...),
#     ...
# ])


### Checkpoint 1.4: Configure Dataset caching and prefetching

Apply `.cache()` and `.prefetch(buffer_size=tf.data.AUTOTUNE)` to your loaded datasets to optimize performance during training.

In [ ]:
# Optimize dataset loaders for performance
AUTOTUNE = tf.data.AUTOTUNE

# train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
# val_ds = ...
# test_ds = ...


# Phase 2: Model Architectures

In this phase, you will define your model architectures. We will create two options so you can compare their performance:
1. **Custom CNN from scratch:** To learn how standard convolution and pooling layers extract spatial patterns.
2. **Transfer Learning (MobileNetV2):** Using pre-trained feature extractors trained on ImageNet.

### Checkpoint 2.1: Build a Custom CNN from Scratch

Create a CNN model using `tf.keras.Sequential`.

**Guidelines:**
- Start with a `layers.Rescaling(1./255)` layer to normalize pixel values from `[0, 255]` to `[0, 1]`.
- Add a few convolutional block stages, each with a `Conv2D` layer (using `relu` activation) followed by a `MaxPooling2D` layer.
- Add a `Flatten` layer to convert the 2D feature maps to a 1D vector.
- Add a `Dense` hidden layer with `relu` activation and `Dropout` (e.g., `0.5`) to combat overfitting.
- Add the final classification `Dense` layer with **3 output units** and a **`softmax`** activation function.

In [ ]:
from tensorflow.keras import layers, models

def build_custom_cnn(input_shape=(224, 224, 3)):
    model = models.Sequential([
        # Todo: 1. Add Rescaling layer
        # Todo: 2. Add Conv2D & MaxPooling2D layers
        # Todo: 3. Add Flatten & Dense layer
        # Todo: 4. Add Final Dense layer with 3 units (softmax)
    ])
    return model

custom_model = build_custom_cnn()
custom_model.summary()


### Checkpoint 2.2: Build a Transfer Learning Model (MobileNetV2)

Create a model utilizing a pre-trained `MobileNetV2` base model from `tf.keras.applications`.

**Guidelines:**
1. Instantiate `MobileNetV2` with `include_top=False`, `weights='imagenet'`, and `input_shape=(224, 224, 3)`.
2. Freeze the pre-trained base model by setting `base_model.trainable = False`.
3. Create a sequential wrapper combining: 
   - A preprocessing function layer specifically for MobileNetV2: `tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input)`
   - Your pre-trained base model.
   - `layers.GlobalAveragePooling2D()` to pool the 2D feature maps.
   - A `layers.Dropout(0.3)` layer.
   - A final `layers.Dense(3, activation='softmax')` classifier.

In [ ]:
def build_transfer_model(input_shape=(224, 224, 3)):
    # Load MobileNetV2 base model
    # base_model = tf.keras.applications.MobileNetV2(...)
    
    # Freeze base model
    # base_model.trainable = False
    
    # Build the wrapper model
    # model = models.Sequential([
    #     layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input, input_shape=input_shape),
    #     base_model,
    #     layers.GlobalAveragePooling2D(),
    #     layers.Dropout(0.3),
    #     layers.Dense(3, activation='softmax')
    # ])
    pass

# transfer_model = build_transfer_model()
# transfer_model.summary()


# Phase 3: Model Compilation and Training

In this phase, you will configure your model for training and run it.

### Checkpoint 3.1: Compile the Model

Choose which model you want to train first (e.g., `transfer_model` or `custom_model`). Compile it using:
- **Optimizer:** `tf.keras.optimizers.Adam` with a learning rate of `1e-4`.
- **Loss:** `'categorical_crossentropy'` (since our dataset uses categorical label mode).
- **Metrics:** `['accuracy']`.

In [ ]:
# Choose model
# model_to_train = transfer_model

# Compile model
# model_to_train.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )


### Checkpoint 3.2: Configure Callbacks and Train

Train the model using `model.fit()`. Use the following Keras callbacks to make your training robust:
1. **`EarlyStopping`:** Set `monitor='val_loss'`, `patience=5`, and `restore_best_weights=True`.
2. **`ModelCheckpoint`:** Save the best weights to a file (e.g., `'best_model.keras'`) by setting `monitor='val_loss'` and `save_best_only=True`.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Define callbacks
# callbacks = [
#     EarlyStopping(...),
#     ModelCheckpoint(...)
# ]

EPOCHS = 20

# Train
# history = model_to_train.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS,
#     callbacks=callbacks
# )
